# 02 — Features, labels, and the feature space

*Module 00 · Getting Started — notebook 2 of 11*

In notebook 01 we met features, labels, and examples. Here we make the picture precise — data as a cloud of **points** — and pick up two small tools we'll lean on again and again.

**Prerequisites:** 01 (what is machine learning?).

**What you'll be able to do**
- Lay data out as a feature matrix `X` and a label vector `y`, and name the kinds of features.
- See each example as a **point** in feature space.
- Compute the **mean (centroid)** of a group of points.
- Compute the **Euclidean distance** between two points, and say what "closest" means.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ml_course import colors, datasets, viz

np.random.seed(0)
viz.use_course_style()

# The two species and the two features we work with throughout this module.
SPECIES_ORDER = ["Adelie", "Gentoo"]
FEATURES = datasets.PENGUINS_FEATURES  # ['bill_length_mm', 'flipper_length_mm']

## Where we are

From notebook 01 we have the vocabulary: each penguin is an **example**, its measurements are **features**, and its species is the **label**. We could already read the scatter and guess a species by eye.

Now we turn that informal picture into something we can compute with. Two tools will carry us a long way — the **mean** of a group of points, and the **distance** between two points — and in notebook 05 they become our first real classifier.

## The feature matrix `X` and the label vector `y`

By convention, we arrange data in two pieces:

- **`X`** — the *feature matrix*: one **row per example**, one **column per feature**. With 274 penguins and 2 measurements, `X` has shape (274, 2).
- **`y`** — the *label vector*: one entry per example, holding the answer we want to predict (the species).

Almost every tool in this course expects exactly this `X`, `y` pair. Let's build it from the penguins frame.

In [ ]:
X, y = datasets.penguins_xy()

print("X (features):", X.shape, "->  n_samples =", X.shape[0], ", n_features =", X.shape[1])
print("y (label)   :", y.shape)
print("first labels:", list(y.head().values))

X.head()

### Read the output

`X` is a table of shape **(274, 2)** — 274 rows (`n_samples`) and 2 columns (`n_features`, here bill and flipper length). `y` is a matching list of 274 species labels, lined up row for row with `X`: the label `y[i]` is the answer for the features in row `i` of `X`. We call `X` the **design matrix** (n × d, for n examples and d features) and `y` the **target**.

## Kinds of features

Not every feature is a number. It helps to name the common kinds:

- **Numeric** — a measured quantity, like our bill length and flipper length (in millimetres).
- **Categorical** — a label with no inherent order, like the species or the island a penguin lives on.
- **Ordinal** — categories with an order but no fixed spacing, like "small / medium / large".

Our two features are numeric, which is what lets us place each penguin at a position along an axis. Categorical features (like the label itself) need to be turned into numbers before most methods can use them — a step we meet in notebook 11.

## Each example is a point

Here is the idea that makes the rest of the course visual: an example with two numeric features *is* a point. Its bill length is its position along the horizontal axis, its flipper length its position along the vertical axis. The whole dataset becomes a **cloud of points** — the **feature space**.

With two features we can draw that cloud on a page. With three we could build it in a box. Beyond three, we can no longer see all of it at once — a limitation we'll learn to work around later. For now, two features keep everything in plain view.

In [ ]:
one = X.iloc[0]  # look at the very first penguin as a single point

fig, ax = plt.subplots(figsize=(7, 5.5))
for species, color in zip(SPECIES_ORDER, colors.CLASS_CYCLE):
    sub = X[y == species]
    ax.scatter(
        sub[FEATURES[0]],
        sub[FEATURES[1]],
        color=color,
        edgecolor=colors.COLORS["text"],
        linewidth=0.5,
        s=40,
        label=species,
        alpha=0.7,
    )

ax.scatter(
    one[FEATURES[0]],
    one[FEATURES[1]],
    color=colors.COLORS["highlight"],
    edgecolor=colors.COLORS["text"],
    s=170,
    zorder=5,
    label="one penguin",
)
ax.annotate(
    f"({one[FEATURES[0]]:.1f}, {one[FEATURES[1]]:.0f})",
    (one[FEATURES[0]], one[FEATURES[1]]),
    textcoords="offset points",
    xytext=(10, 8),
    color=colors.COLORS["text"],
)

ax.set_xlabel(FEATURES[0])
ax.set_ylabel(FEATURES[1])
ax.set_title("Each penguin is a point in feature space")
ax.legend(title="species")
plt.show()

### Read the figure

This is the same scatter as in notebook 01, now read as pure geometry: every penguin is a point, and the highlighted one carries its coordinates — its two feature values — written beside it. The position *is* the data. Holding that equivalence — row of numbers ↔ point in space — is what lets us measure things about the data geometrically, which is what we do next.

## The mean: the middle of a cloud

The first thing we can measure about a group of points is where its **middle** sits. The **mean** (or **centroid**) of a set of points is found feature by feature: average all the bill lengths to get the middle's horizontal position, average all the flipper lengths to get its vertical position. The result is itself a point — the balance point of the cloud.

We can take the mean of the whole dataset, or of one species at a time. Let's do both.

In [ ]:
overall_mean = X.mean()
class_means = X.groupby(y).mean()

print("Overall mean point:")
print(overall_mean.to_string())
print("\nPer-species mean (centroid):")
print(class_means.to_string())

fig, ax = plt.subplots(figsize=(7, 5.5))
for species, color in zip(SPECIES_ORDER, colors.CLASS_CYCLE):
    sub = X[y == species]
    ax.scatter(
        sub[FEATURES[0]],
        sub[FEATURES[1]],
        color=color,
        edgecolor="none",
        s=30,
        alpha=0.35,
        label=species,
    )
    m = class_means.loc[species]
    ax.scatter(
        m[FEATURES[0]],
        m[FEATURES[1]],
        color=color,
        edgecolor=colors.COLORS["text"],
        linewidth=1.0,
        s=280,
        marker="X",
        zorder=5,
        label=f"{species} mean",
    )

ax.set_xlabel(FEATURES[0])
ax.set_ylabel(FEATURES[1])
ax.set_title("Each species and its mean point (centroid)")
ax.legend(title="species")
plt.show()

### Read the figure

Each large marker is the **mean point** of one species — and it sits right in the middle of that species' cloud, exactly as a balance point should. Notice that the two means are **clearly apart**: Gentoos average longer flippers and larger bills than Adélies.

Hold that thought. If the two groups have well-separated middles, then "which mean is a new penguin closest to?" is a reasonable way to guess its species. Turning that idea into our first classifier is exactly what notebook 05 does — but to ask "closest", we first need to say what distance means.

## Distance between two points

"Closest" needs a ruler. The everyday one is the **Euclidean distance** — the straight-line distance between two points. In two dimensions it is the Pythagorean theorem: the distance between points $a = (a_1, a_2)$ and $b = (b_1, b_2)$ is

$$ d(a, b) = \sqrt{(a_1 - b_1)^2 + (a_2 - b_2)^2}. $$

The same formula extends to any number of features — sum the squared differences across all of them, then take the square root. Let's compute one distance by hand and check it.

In [ ]:
a = X.iloc[0]   # an Adelie penguin
b = X.iloc[-1]  # a Gentoo penguin

diff = a.to_numpy() - b.to_numpy()
dist_by_hand = float(np.sqrt((diff**2).sum()))
dist_numpy = float(np.linalg.norm(a.to_numpy() - b.to_numpy()))

print(f"penguin A (bill, flipper) = ({a[FEATURES[0]]:.1f}, {a[FEATURES[1]]:.0f})")
print(f"penguin B (bill, flipper) = ({b[FEATURES[0]]:.1f}, {b[FEATURES[1]]:.0f})")
print(f"distance by hand : {dist_by_hand:.2f}")
print(f"numpy norm       : {dist_numpy:.2f}")

fig, ax = plt.subplots(figsize=(7, 5.5))
for species, color in zip(SPECIES_ORDER, colors.CLASS_CYCLE):
    sub = X[y == species]
    ax.scatter(
        sub[FEATURES[0]], sub[FEATURES[1]], color=color, edgecolor="none", s=30, alpha=0.35, label=species
    )

ax.plot(
    [a[FEATURES[0]], b[FEATURES[0]]],
    [a[FEATURES[1]], b[FEATURES[1]]],
    color=colors.COLORS["text"],
    linewidth=1.5,
    marker="o",
)
ax.annotate(
    f"distance = {dist_by_hand:.1f}",
    ((a[FEATURES[0]] + b[FEATURES[0]]) / 2, (a[FEATURES[1]] + b[FEATURES[1]]) / 2),
    textcoords="offset points",
    xytext=(8, 8),
    color=colors.COLORS["text"],
)

ax.set_xlabel(FEATURES[0])
ax.set_ylabel(FEATURES[1])
ax.set_title("Euclidean distance between two penguins")
ax.legend(title="species")
plt.show()

### Read the figure

The line connects two penguins, and its length is the Euclidean distance we computed — the by-hand value and NumPy's agree, a small reassurance that the formula and the code say the same thing.

One honest caveat before we move on. That single number mixes millimetres of bill length with millimetres of flipper length. The Euclidean distance does not care about a feature's overall range — it adds up the *differences* along each axis — and here the differences along the flipper axis happen to be numerically larger, so they tend to dominate the total. A fixed gap in bill length then counts for less than the same gap in flipper length, even though the two may matter equally. In other words, **distance depends on the scale of the features**. We don't resolve this today — we name it, and return to it deliberately in notebook 11.

## The words we'll keep using

Added this notebook:

- **Feature matrix `X`** — rows are examples, columns are features; shape (n_samples, n_features).
- **Label vector `y`** — one target per example, aligned with `X`.
- **Design matrix** — another name for `X` (n × d).
- **Feature type** — numeric, categorical, or ordinal.
- **Feature space** — the space whose axes are the features; each example is a point.
- **Mean / centroid** — the middle (balance point) of a group of points.
- **Euclidean distance** — the straight-line distance between two points.

## Your turn

1. Without rerunning the code, state the shape of `X` and the shape of `y`, and say what each number means.
2. By hand, compute the mean point of these three penguins (bill, flipper): (39, 190), (41, 195), (40, 200). Average each feature separately.
3. By hand, compute the Euclidean distance between (39, 190) and (50, 215) — as a check, it should come out close to 27. Then say which of (39, 190) or (50, 215) is closer to a new penguin at (46, 210).

Work these on paper or in a new cell. We'll lean on exactly these two operations — mean and distance — in notebook 05.

## What you built

You now have the geometric toolkit the rest of this module runs on:

- You can arrange data as a feature matrix `X` and a label vector `y`, and you can name feature types.
- You can read each example as a **point** in feature space.
- You can compute the **mean (centroid)** of a group of points — a whole class, or all of it.
- You can measure the **Euclidean distance** between two points, and you've seen that distance depends on feature scale.

Mean and distance look modest on their own. In notebook 05 they combine into a complete, working classifier — your first. Well done getting the pieces in place.

## References

- A. Géron, *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow*, 3rd ed., O'Reilly, 2022 — chapter 2 (working with real data).
- G. James, D. Witten, T. Hastie, R. Tibshirani, *An Introduction to Statistical Learning*, 2nd ed., Springer, 2021 — §2.1. DOI: 10.1007/978-1-0716-1418-1

---
Previous: **01 — What is machine learning?** · Next: **03 — Look before you model**